Total training compute(in GFLOPs) for models found on the pareto frontier(Scaling laws for robust comparison).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

C = np.array([7.05e+6, 1.69e+7, 3.5e+7, 7.05e+7, 1.24e+8, 1.69e+8])  # considering first 7 models
Cmin, Cmax = C.min(), C.max()

target_logs = np.linspace(np.log10(Cmin), np.log10(Cmax), 5)

plt.figure()
plt.xscale("log")
plt.scatter(np.log10(C), np.zeros_like(C), label="Actual models")
plt.scatter(target_logs, np.zeros_like(target_logs), marker="x", label="Target log points")

plt.yticks([])
plt.xlabel("log10(GFLOPs)")
plt.legend()
plt.title("Actual compute vs. 5 target log-spaced baseline points")

plt.show()

C = np.array([7.05e+6, 1.5e+7, 1.69e+7, 3.5e+7, 7.05e+7, 1.24e+8, 1.69e+8, 3.52e+8,
              6.23e+8])  # considering first 9 models, assuming we have 64m samples
Cmin, Cmax = C.min(), C.max()

target_logs = np.linspace(np.log10(Cmin), np.log10(Cmax), 5)
plt.figure()
plt.xscale("log")
plt.scatter(np.log10(C), np.zeros_like(C), label="Actual models")
plt.scatter(target_logs, np.zeros_like(target_logs), marker="x", label="Target log points")

plt.yticks([])
plt.xlabel("log10(GFLOPs)")
plt.legend()
plt.title("Actual compute vs. 5 target log-spaced baseline points")

plt.show()

Baseline results plotting

In [ ]:
from train.utils import plot_compute_vs_metric, Plotting

plot_compute_vs_metric("runs_config.csv", Plotting.VALIDATION_ERROR)
plot_compute_vs_metric("runs_config.csv", Plotting.TEST_ERROR)
plot_compute_vs_metric("runs_config.csv", Plotting.IMAGENET)
plot_compute_vs_metric("runs_config.csv", Plotting.MSCOCO_IMG)


Load CLIP model with custom vision and text tower

In [ ]:
from train.utils import precision_to_dtype
from train.clip_train.profiler import get_model_flops
from open_clip import create_model

model = create_model("local-dir:model_configs/ViT-B-32", precision="bf16")

get_model_flops(model, 100, dtype=precision_to_dtype("bf16"))


Batch size heuristic based on pareto frontier

In [ ]:

from train.utils import monotonic_bs, query_bs
import numpy as np

samples_seen = np.array([
    1.28e+06, 3.07e+06, 6.4e+06, 1.28e+07,
    3.07e+07, 6.4e+07, 1.28e+08, 3.07e+08,
    6.4e+08, 1.28e+09, 3.07e+09
], dtype=float)

batch_size = np.array([
    512, 1024, 1024, 2048, 4096, 4096,
    8192, 16384, 32768, 90624, 91136
], dtype=int)

ranges = monotonic_bs(samples_seen, batch_size)
print(query_bs(705280005))


Infer hardware requirements

In [ ]:
import math

global_batch_size = 90624
local_bs = 128
max_gpus_per_node = 4

gpus_needed = global_batch_size // local_bs
gpus_per_node = min(gpus_needed, max_gpus_per_node)
nodes_nr = math.ceil(gpus_needed / max_gpus_per_node)

print(f"GPUs needed:{gpus_needed}")
print(f"GPUs per node:{gpus_per_node}")
print(f"Nodes number:{nodes_nr}")

In [ ]:
import neps

root_dir = "results/model=ViT-S-32/seed=0"
full_df, result = neps.status(root_dir)
print(full_df)


Max. amount of GFLOPs for our case

In [1]:
from train.utils import precision_to_dtype
from train.clip_train.profiler import get_model_flops
from open_clip import create_model

def count_params(model):
    return sum(m.numel() for m in model.parameters())

nr_defined_samples = 130363082
model = create_model("local-dir:model_configs/ViT-S-32", precision="amp")

flops_amount = get_model_flops(model, nr_defined_samples, dtype=precision_to_dtype("amp"))
print(flops_amount)
print(count_params(model))


C:\Users\Donat\miniconda3\envs\vlm_scaling_law\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Module                     FLOP    % Total
--------------------  ---------  ---------
CLIP                  7476.511M    100.00%
 - aten.convolution    134.873M      1.80%
 - aten.addmm         7340.950M     98.19%
 - aten.mm               0.688M      0.01%
 CLIP.transformer     4450.812M     59.53%
  - aten.addmm        4450.812M     59.53%
 CLIP.visual          3025.355M     40.46%
  - aten.convolution   134.873M      1.80%
  - aten.addmm        2890.138M     38.66%
  - aten.mm              0.344M      0.00%
975115853.36
81857665


In [ ]:
from train.utils import build_model_cfg

build_model_cfg(text_width=256, vision_width=413,
                out_path="results/model=ViT-S-32/optimizer=experimental/seed=1/model_cfg/config_1")

GFLOPs to GPU hours mapping for RTX2080s and L40s

In [ ]:
from train.utils import add_gpu_hours_column, plot_measured_vs_predicted_gpu_hours
from train.flops_to_gpu_hours import GpuHoursModel
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

bs_rules = [
    {"min": 0.0, "max": 16915705.0, "bs": 512},
    {"min": 16915705.0, "max": 70528005.0, "bs": 1024},
    {"min": 70528005.0, "max": 169157005.0, "bs": 2048},
    {"min": 169157005.0, "max": 705280005.0, "bs": 4096},
    {"min": 705280005.0, "max": 1691570005.0, "bs": 8192},
]

new_bs_rules = [
    {"min": 0.0, "max": 1459000.0, "bs": 512},
    {"min": 1459000.0, "max": 7040000.0, "bs": 1024},
    {"min": 7040000.0, "max": 14590000.0, "bs": 2048},
    {"min": 14590000.0, "max": 70400000.0, "bs": 4096},
    {"min": 70400000.0, "max": 145900000.0, "bs": 8192},
]

PEAK_GFLOPS = {
    "RTX2080": 10300,
    "L40s": 90500,
    "NVIDIA A100-SXM4-40GB": 156000,
}

df = pd.concat(
    [
        pd.read_csv("runs_csvs/scale_space_only_search_seed_42.csv"),
        pd.read_csv("runs_csvs/scale_space_only_search_biased_sampling_seed_42.csv"),
    ],
    ignore_index=True,
)

A100_df = pd.concat(
    [
        pd.read_csv("results/scale_space_only_search_juwels/seed=40/configs_summary.csv"),
        pd.read_csv("results/scale_space_only_search_biased_scale_juwels/seed=40/configs_summary.csv"),
        pd.read_csv("results/scan_space_juwels_biased_scale_alpha_0_5/configs_summary.csv"),
    ],
    ignore_index=True,
)

df = add_gpu_hours_column(df)
A100_df = add_gpu_hours_column(A100_df)

x_query = np.logspace(7, 9, 6)

for gpu_name in ["RTX2080", "L40s"]:
    model = GpuHoursModel.fit_from_df(
        df,
        gpu_name,
        bs_rules=bs_rules,
    )

    plot_measured_vs_predicted_gpu_hours(
        df,
        gpu_name=gpu_name,
        model=model,
        x_query=x_query,
    )

A100_gpu_name = "NVIDIA A100-SXM4-40GB"
A100_model = GpuHoursModel.fit_from_df(
    A100_df,
    A100_gpu_name,
    bs_rules=new_bs_rules,
    use_compute_col=False
)

plot_measured_vs_predicted_gpu_hours(
    A100_df,
    gpu_name=A100_gpu_name,
    model=A100_model,
    x_query=x_query,
)

plt.show()

GFLOPs distribution for different sampling strategies

In [ ]:
from train.flops_to_gpu_hours import SamplingStrategyPlotter
import pandas as pd
import numpy as np


def gpu_hours_model():
    A100_gpu_name = "NVIDIA A100-SXM4-40GB"
    A100_df = pd.concat(
    [
        pd.read_csv("results/scale_space_only_search_juwels/seed=40/configs_summary.csv"),
        pd.read_csv("results/scale_space_only_search_biased_scale_juwels/seed=40/configs_summary.csv"),
        pd.read_csv("results/scan_space_juwels_biased_scale_alpha_0_5/configs_summary.csv"),
    ],
    ignore_index=True,
   )

    A100_model = GpuHoursModel.fit_from_df(
        A100_df,
        A100_gpu_name,
        bs_rules=new_bs_rules,
        use_compute_col=False
    )

    return A100_model


# uniform sampling in individual parameters does not imply uniform sampling in compute space!


dfs = {
    "random sampling": pd.read_csv("sampling_strategy_feat_distributions/random_sampling.csv"),
    "biased_sampling, alpha 1": pd.read_csv("sampling_strategy_feat_distributions/biased_sampling_alpha_1.csv"),
    "biased sampling, alpha 0.5": pd.read_csv("sampling_strategy_feat_distributions/biased_sampling_alpha_0_5.csv")
}

plotter = SamplingStrategyPlotter(dfs)

plotter.plot_default_suite(
    compute_col="total_compute",
    vision_col="vision_width",
    text_col="text_width",
    train_samples_col="train_num_samples",
    bins_compute=25,
    bins_train=25,
    title="Feature distributions per sampling strategy",
    density=False,
)

gflops_mean_per_strategy = {}
model = gpu_hours_model()

target_Ns = [100, 500, 1000]

for name, df in dfs.items():
    gflops = pd.to_numeric(df["total_compute"], errors="coerce").to_numpy(dtype=float)
    gflops = gflops[np.isfinite(gflops) & (gflops > 0)]

    if gflops.size == 0:
        print(f"\n{name}\nNo valid total_compute values.")
        continue

    mean_gflops = float(np.mean(gflops))

    mean_gpu_hours_per_cfg = float(
        model.mapping_gflops_to_gpu_hours(np.array([mean_gflops]), mode="global_linear", as_dict=False).sum()
    )

    print(f"\n{name}")
    print(f"Mean GFLOPs per config: {mean_gflops:.4f}")
    print(f"Mean GPU hours per config: {mean_gpu_hours_per_cfg:.6f}")

    for N in target_Ns:
        total_gpu_hours = N * mean_gpu_hours_per_cfg
        print(f"Estimated GPU hours for N={N}: {total_gpu_hours:.4f}")

GPU_hours_left = 28000


In [ ]:
# Different possibilities
# 1. Inverse transform sampling(To map uniform random values to any target distribution, you use the inverse CDF of that distribution) of values from uniform distribution to exponential(lambda) distribution. We need to truncate and shift the distribution.
# 2. sample directly from a log distribution. Same logic by finding the inverse CDF.


import numpy as np
import matplotlib.pyplot as plt

lower = 0.6
upper = 60.0

x = np.linspace(lower, upper, 2000)

pdf_linear = np.ones_like(x) / (upper - lower)
pdf_log = 1.0 / (x * np.log(upper / lower))

plt.figure(figsize=(8, 5))
plt.plot(x, pdf_linear, label="Linear sampling", linewidth=2)
plt.plot(x, pdf_log, label="Log sampling (∝ 1/x)", linewidth=2)

plt.xlabel("train_num_samples (millions)")
plt.ylabel("Probability density")
plt.title("Probability density: linear vs log sampling")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()


Warped log-uniform distribution

In [ ]:
# Different possibilities:
# 1. Inverse transform sampling(To map uniform random values to any target distribution, you use the inverse CDF of that distribution) of values from uniform distribution to exponential(lambda) distribution. We need to truncate and shift the distribution.
# 2. sample directly from a log distribution. Same logic by finding the inverse CDF. Could have another warping param.

import numpy as np
import matplotlib.pyplot as plt

lower = 0.6
upper = 60.0
n = 20000

# alpha = 1.0 pure log-uniform
# alpha > 1.0 increasingly biased towards the lower bound
alpha = 1.0

rng = np.random.default_rng(42)
u = rng.uniform(0.0, 1.0, size=n)

samples = lower * (upper / lower) ** (u ** alpha)

plt.figure()
plt.scatter(np.arange(1, n + 1), samples)
plt.xlabel("Sample index")
plt.ylabel("train_num_samples (millions)")
plt.title(f"Samples biased toward small values (alpha={alpha})")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure()
plt.hist(samples, bins=20)
plt.xlabel("train_num_samples (millions)")
plt.ylabel("Count")
plt.title(f"Histogram of 100 biased samples (alpha={alpha})")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import neps
import matplotlib.pyplot as plt

full_df, result = neps.status("results/optimizer=experimental/seed=0")

vision = full_df["config.vision_width"]
text = full_df["config.text_width"]
train_samples = full_df["config.train_num_samples"]

plt.hist(vision, bins=range(int(vision.min()), int(vision.max()) + 2), edgecolor="black")
plt.xlabel("vision_width")
plt.ylabel("count")
plt.title("Distribution of vision_width")
plt.show()

plt.hist(text, bins=range(int(text.min()), int(text.max()) + 2), edgecolor="black")
plt.xlabel("text_width")
plt.ylabel("count")
plt.title("Distribution of text_width")
plt.show()

plt.hist(train_samples, bins=30)
plt.xscale("log")
plt.xlabel("train_num_samples")
plt.ylabel("count")
plt.title("Distribution of train_num_samples (log scale)")
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("results/scale_space_only_search_juwels/seed=40/configs_summary.csv")

df["train_num_samples"] = (
        pd.to_numeric(df["train_num_samples"], errors="coerce") * 1_000_000
).round().astype("Int64")

df.to_csv("results/scale_space_only_search_juwels/seed=40/configs_summary.csv", index=False)

In [ ]:
from train.scaling_law_plots.plotting import plot_all_objectives

plot_all_objectives("results/juwels_joint_results/configs_summary.csv")


In [ ]:
from train.utils import sample_small_favoring_index
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

N = 10_000
n = 9
alpha = 1.0

rng = np.random.default_rng(42)

uniform_samples = rng.integers(0, n, size=N)

warped_samples = np.array([sample_small_favoring_index(n=n, alpha=alpha, rng=rng) for _ in range(N)])

uniform_counts = Counter(uniform_samples)
warped_counts = Counter(warped_samples)

x = np.arange(n)
uniform_freq = np.array([uniform_counts[i] for i in x]) / N
warped_freq = np.array([warped_counts[i] for i in x]) / N

plt.figure(figsize=(8, 4))
plt.bar(x - 0.2, uniform_freq, width=0.4, label="uniform input")
plt.bar(x + 0.2, warped_freq, width=0.4, label=f"biased output (alpha={alpha})")
plt.xlabel("index")
plt.ylabel("probability")
plt.title("Discrete small-favoring sampling distribution")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("results/scan_space_juwels_biased_scale_alpha_0_5/configs_summary.csv")
df["experiment"] = "scale space search, biased scale space, alpha 0.5"
df.to_csv("file.csv", index=False)

Calculate the optimal n_samples and model size given GFLOPs of 1e9

In [6]:
import numpy as np
compute = 1e9

n_datapoints = 5.73e4 * np.power(compute, 0.373)
print(n_datapoints)

n_params = 3.24e3 * np.power(compute, 0.483)
print(n_params)

130363082.78323963
72035240.45710544
